<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/SunnyChoudhary850/FlyRank/main/data/raw/content_refresh_anonymized.csv")
df.shape
df.head(5)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Lane: **Refresh / Content Opportunity Scoring**

**Task type: Classification, used for scoring/ranking.**

The core task is a binary classification problem: is this page declining or not? But we
don't just want a yes/no answer — we want a *probability score* per page, so pages can be
ranked from "most likely needs review" to "least likely." That ranked list is what a content
team actually uses, so this is classification feeding into a ranking output.

In [8]:
df['declining_label'] = (df['trend_direction'] == 'down')
df[['content_id', 'trend_direction', 'declining_label']].head(10)

,content_id,trend_direction,declining_label
0,content_304f48230142,down,True
1,content_a1fb4e703a9e,down,True
2,content_9aa793d4d895,down,True
3,content_331d6c4de07b,stable,False
4,content_d99b7a2d90ca,down,True
5,content_d4084a4bc775,down,True
6,content_9a34b442b552,down,True
7,content_a63219c6e95a,stable,False
8,content_5e6c160719bc,down,True
9,content_c27558df2b0c,down,True


In [9]:
df['declining_label'].value_counts()
df['declining_label'].value_counts(normalize=True)

,proportion
declining_label,
True,0.542067
False,0.457933


In [14]:
unit_of_analysis = df[['content_id', 'client_id', 'content_type', 'word_count',
                        'days_since_last_update', 'declining_label']].copy()
unit_of_analysis.head(10)

,content_id,client_id,content_type,word_count,days_since_last_update,declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,20,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,25,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,20,True
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,22,False
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,14,True
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3080.0,20,True
6,content_9a34b442b552,client_8722616204,keyword article,3059.0,20,True
7,content_a63219c6e95a,client_19581e27de,keyword article,NaN,22,False
8,content_5e6c160719bc,client_6208ef0f77,keyword article,3807.0,20,True
9,content_c27558df2b0c,client_19581e27de,keyword article,NaN,104,True


In [15]:
unit_of_analysis[['content_type', 'word_count', 'days_since_last_update']].isnull().sum()

,0
content_type,0
word_count,7699
days_since_last_update,0


**Note on missing data:** `word_count` is missing for 7,699 rows — likely tied to specific
content types (e.g. feed-based articles that don't track word count at all), not random
gaps. This means a blind fillna(0) would silently treat "no word count recorded" the same
as "zero words," which isn't true. Any model using this feature needs to check missingness
per content_type first, rather than filling blindly.